# S5 · De una función a una interfaz Streamlit
## Guion docente de ejemplos · Churn sintético
**7 de octubre de 2026 · Operación de Modelos · MUIAAP**

Objetivo: ver cómo un script se convierte en una interfaz y distinguir **editar** de **enviar**. El caso y el contrato siguen la práctica 5.1 del repositorio actual, no la antigua práctica Wine de las diapositivas.

Este notebook es el guion. Las celdas preparan archivos `.py`; **la interfaz se ve en el navegador**, no dentro de Jupyter. No hace falta entrenar, descargar datos, llamar una API ni configurar credenciales.

Recorrido sugerido (30-40 min de demostraciones, intercalado con la teoría): arranque → widgets y rerun → formulario → predicción y error. Después: práctica 5.1 del repo. Estado, caché y telemetría quedan para S6.

## 0. Preparación (5 min)
Abre este notebook en tu editor con un kernel Python 3.11 o posterior. Ejecuta las celdas de arriba abajo. Solo usan la biblioteca estándar para escribir archivos.

Si falta kernel en VS Code, en una carpeta de trabajo propia, desde **Git Bash**:
```bash
uv init --python 3.12
uv add --dev ipykernel
```
Selecciona el Python de `.venv` como kernel. Si ya tienes kernel, no repitas la instalación.

Los ejemplos se guardan en una carpeta nueva del directorio del kernel. No modifican el starter de los alumnos. Guarda este notebook antes de empezar. La celda siguiente imprime la ruta exacta.

In [1]:
from pathlib import Path
from datetime import datetime
import sys

assert sys.version_info >= (3, 11), "Selecciona un kernel Python 3.11 o posterior."
DEMO_DIR = Path.cwd() / ("demo_streamlit_s5_" + datetime.now().strftime("%Y%m%d_%H%M%S_%f"))
DEMO_DIR.mkdir()
print("Carpeta de los ejemplos:", DEMO_DIR.resolve())


Carpeta de los ejemplos: C:\Users\pedro\OneDrive\Escritorio\MASTER\Operación de Modelos\Repos_(clon_de_mi_fork)\icai-muiaap-operacion-de-modelos\semana5\demo_streamlit_s5_20261007_200308_036408


### Contrato de inferencia ya resuelto
La función original del repo recibe:
- `tenure_months`: entero entre 0 y 120.
- `monthly_spend_eur`: número entre 0 y 300.
- `support_calls`: entero entre 0 y 20.
- `has_annual_contract`: booleano.

Devuelve `will_churn`, `label`, `risk_score` y `explanation`. **Es una regla sintética, no un modelo entrenado; el score no es una probabilidad calibrada.** Copiamos la función sin cambiarla y separamos inferencia de interfaz.

In [ ]:
MODEL_SOURCE = '''"""Contrato de inferencia consumido por la interfaz Streamlit de la práctica."""

from math import isfinite
from typing import Literal, TypedDict

ChurnLabel = Literal["Baja probable", "Permanencia probable"]


class ChurnPrediction(TypedDict):
    """Salida pequeña, serializable y comprensible por cualquier interfaz."""

    will_churn: bool
    label: ChurnLabel
    risk_score: float
    explanation: str


def _whole_number(name: str, value: int | float, upper: int) -> int:
    """Normaliza los enteros que un slider puede entregar como `float`."""

    if isinstance(value, bool) or not isinstance(value, (int, float)):
        raise ValueError(f"{name} debe ser un número entero")
    number = float(value)
    if not isfinite(number) or not number.is_integer():
        raise ValueError(f"{name} debe ser un número entero")
    normalized = int(number)
    if not 0 <= normalized <= upper:
        raise ValueError(f"{name} debe estar entre 0 y {upper}")
    return normalized


def _bounded_number(name: str, value: int | float, upper: float) -> float:
    """Valida un número real finito dentro del rango didáctico."""

    if isinstance(value, bool) or not isinstance(value, (int, float)):
        raise ValueError(f"{name} debe ser un número")
    normalized = float(value)
    if not isfinite(normalized) or not 0 <= normalized <= upper:
        raise ValueError(f"{name} debe estar entre 0 y {upper:g}")
    return normalized


def predict(
    tenure_months: int | float,
    monthly_spend_eur: int | float,
    support_calls: int | float,
    has_annual_contract: bool,
) -> ChurnPrediction:
    """Aplica una regla sintética reproducible; no es un modelo entrenado."""

    tenure = _whole_number("tenure_months", tenure_months, 120)
    monthly_spend = _bounded_number("monthly_spend_eur", monthly_spend_eur, 300)
    calls = _whole_number("support_calls", support_calls, 20)
    if not isinstance(has_annual_contract, bool):
        raise ValueError("has_annual_contract debe ser booleano")

    score = 0.25
    factors = ["base: +0,25"]

    if tenure < 6:
        score += 0.30
        factors.append("antigüedad inferior a 6 meses: +0,30")
    elif tenure >= 24:
        score -= 0.15
        factors.append("antigüedad de al menos 24 meses: -0,15")
    else:
        factors.append("antigüedad entre 6 y 23 meses: +0,00")

    if monthly_spend >= 80:
        score += 0.20
        factors.append("gasto mensual de al menos 80 €: +0,20")
    elif monthly_spend < 40:
        score -= 0.05
        factors.append("gasto mensual inferior a 40 €: -0,05")
    else:
        factors.append("gasto mensual entre 40 y 79,99 €: +0,00")

    if calls >= 3:
        score += 0.25
        factors.append("tres o más llamadas de soporte: +0,25")
    elif calls >= 1:
        score += 0.10
        factors.append("una o dos llamadas de soporte: +0,10")
    else:
        factors.append("sin llamadas de soporte: +0,00")

    if has_annual_contract:
        score -= 0.25
        factors.append("contrato anual: -0,25")
    else:
        score += 0.10
        factors.append("sin contrato anual: +0,10")

    risk_score = round(min(0.95, max(0.05, score)), 2)
    will_churn = risk_score >= 0.50
    label: ChurnLabel = "Baja probable" if will_churn else "Permanencia probable"
    score_text = f"{risk_score:.2f}".replace(".", ",")
    explanation = (
        "Regla docente: "
        + "; ".join(factors)
        + f". Score acotado: {score_text}; umbral de decisión: 0,50. "
        "No es una probabilidad calibrada."
    )

    return {
        "will_churn": will_churn,
        "label": label,
        "risk_score": risk_score,
        "explanation": explanation,
    }
'''
(DEMO_DIR / "model.py").write_text(MODEL_SOURCE, encoding="utf-8")
print("Contrato guardado en model.py")


In [ ]:
import importlib.util
spec = importlib.util.spec_from_file_location("s5_demo_model", DEMO_DIR / "model.py")
model = importlib.util.module_from_spec(spec)
spec.loader.exec_module(model)
predict = model.predict

profiles = [(2, 95, 4, False), (36, 35, 0, True)]
for profile in profiles:
    result = predict(*profile)
    print(profile, "->", result["label"], result["risk_score"])
assert predict(*profiles[0])["risk_score"] == 0.95
assert predict(*profiles[1])["risk_score"] == 0.05


## 1. Primera app (5 min)
`st.title`, `st.caption` y `st.write` dibujan la pantalla. Antes de ejecutar, pregunta: ¿dónde aparecerá este texto?

La celda crea `app_01.py`. No arranca un servidor.

In [ ]:
APP_01 = '''import streamlit as st

st.set_page_config(page_title="S5 · Primera app")
st.title("Mi primera interfaz")
st.caption("Churn sintético · ejemplo docente")
st.write("Python dibuja la pantalla. El navegador permite interactuar.")
'''
(DEMO_DIR / "app_01.py").write_text(APP_01, encoding="utf-8")


### Arranque en Git Bash
Abre otra terminal y entra en **la carpeta que imprimió la primera celda**. En Windows, Git Bash acepta `C:/Users/...` con comillas; también puedes usar su ruta `/c/Users/...`.
```bash
cd "C:/ruta/impresa/demo_streamlit_s5_..."
uv run --python 3.12 --with 'streamlit>=1.40,<2.0' streamlit run app_01.py
```
Abre la URL local que imprime la terminal (normalmente `http://localhost:8501`). La primera ejecución necesita Internet para instalar la dependencia. No ejecutes el comando de servidor en una celda: mantén esa terminal abierta.

Para pasar al ejemplo siguiente: **Ctrl+C**, cambia el nombre del archivo y vuelve a lanzar el mismo comando. Si el puerto está ocupado, añade `--server.port 8502`. No hace falta activar `.venv`.

## 2. Widgets y rerun (7 min)
Añadimos entradas. Cada interacción fuera de un formulario vuelve a ejecutar el script. La hora permite observarlo. Todavía **no llamamos a `predict()`**.

Pregunta: si pusiéramos una inferencia costosa al final del script, ¿cuántas veces se ejecutaría mientras cambiamos tres valores?

In [ ]:
APP_02 = '''from datetime import datetime
import streamlit as st

st.title("Entradas y rerun")
st.caption("Última ejecución: " + datetime.now().isoformat(timespec="microseconds"))
tenure = st.number_input("Antigüedad (meses)", min_value=0, max_value=120, value=2, step=1)
spend = st.number_input("Gasto mensual (EUR)", min_value=0.0, max_value=300.0, value=95.0, step=1.0)
calls = st.number_input("Llamadas a soporte", min_value=0, max_value=20, value=4, step=1)
annual = st.checkbox("Contrato anual", value=False)
st.write({"tenure_months": tenure, "monthly_spend_eur": spend,
          "support_calls": calls, "has_annual_contract": annual})
st.info("Aquí solo recogemos valores. No ejecutamos inferencia.")
'''
(DEMO_DIR / "app_02.py").write_text(APP_02, encoding="utf-8")


Ejecuta `app_02.py` con el mismo comando. Cambia varias entradas y observa la hora y el diccionario. Usa claves técnicas para el contrato y textos claros para la persona. No envíes los textos de los labels al modelo.

## 3. El formulario separa editar de enviar (7 min)
`st.form` agrupa los cambios y `st.form_submit_button` los envía juntos. Editar los campos no ejecuta el script; enviar sí. Solo mostramos el perfil enviado, todavía sin modelo.

Pregunta antes de probar: ¿se actualizará el diccionario mientras escribes o solo al pulsar el botón?

In [ ]:
APP_03 = '''from datetime import datetime
import streamlit as st

st.title("Editar no es enviar")
st.caption("Última ejecución: " + datetime.now().isoformat(timespec="microseconds"))
with st.form("churn_form"):
    tenure = st.number_input("Antigüedad (meses)", min_value=0, max_value=120, value=2, step=1)
    spend = st.number_input("Gasto mensual (EUR)", min_value=0.0, max_value=300.0, value=95.0, step=1.0)
    calls = st.number_input("Llamadas a soporte", min_value=0, max_value=20, value=4, step=1)
    annual = st.checkbox("Contrato anual", value=False)
    submitted = st.form_submit_button("Enviar perfil")

if submitted:
    st.write({"tenure_months": tenure, "monthly_spend_eur": spend,
              "support_calls": calls, "has_annual_contract": annual})
else:
    st.info("Completa el formulario y envíalo.")
'''
(DEMO_DIR / "app_03.py").write_text(APP_03, encoding="utf-8")


Ejecuta `app_03.py`. Edita sin enviar, envía, vuelve a editar y vuelve a enviar. Tras el primer envío, el resultado previo puede seguir visible mientras editas: es el **último perfil enviado**, no una nueva inferencia. No estamos implementando estado persistente.

## 4. Una llamada, un resultado y un error seguro (10 min)
La UI recoge cuatro valores. `predict(**values)` se ejecuta **una sola vez por submit** y nunca mientras se editan los campos. Presentamos `label`, `risk_score` y `explanation` sin recalcular el score.

Una opción fuera del formulario permite provocar un fallo de contrato para enseñar `st.error`. Es solo una herramienta de la demo docente: no forma parte del contrato de la práctica.

In [ ]:
APP_04 = '''import streamlit as st
from model import predict

st.set_page_config(page_title="S5 · Churn sintético")
st.title("¿Qué cliente podría darse de baja?")
st.caption("Regla docente sintética; no es una probabilidad calibrada.")
simulate_error = st.checkbox("Demo docente: provocar un error de contrato", value=False)
with st.form("churn_form"):
    tenure = st.number_input("Antigüedad (meses)", min_value=0, max_value=120, value=2, step=1)
    spend = st.number_input("Gasto mensual (EUR)", min_value=0.0, max_value=300.0, value=95.0, step=1.0)
    calls = st.number_input("Llamadas a soporte", min_value=0, max_value=20, value=4, step=1)
    annual = st.checkbox("Contrato anual", value=False)
    submitted = st.form_submit_button("Calcular riesgo")

if submitted:
    values = {"tenure_months": tenure, "monthly_spend_eur": spend,
              "support_calls": calls, "has_annual_contract": annual}
    if simulate_error:
        values["tenure_months"] = -1
    try:
        result = predict(**values)
    except ValueError:
        st.error("No se pudo calcular el riesgo. Revisa los datos e inténtalo de nuevo.")
    else:
        st.subheader(result["label"])
        st.metric("Score orientativo (0 a 1)", f"{result['risk_score']:.2f}")
        st.write(result["explanation"])
else:
    st.info("Completa el formulario y pulsa Calcular riesgo.")
'''
(DEMO_DIR / "app_04.py").write_text(APP_04, encoding="utf-8")
print("Listo: ejecuta app_04.py en Git Bash.")


### Comprobación en clase
1. Abre la app: hay cuatro campos y no hay predicción.
2. Edita sin enviar: no hay cálculo nuevo.
3. Envía `2, 95, 4, False`: «Baja probable», score 0,95.
4. Envía `36, 35, 0, True`: «Permanencia probable», score 0,05.
5. Activa el fallo docente y envía: mensaje comprensible, sin traceback ni excepción cruda. Desactívalo y vuelve a enviar: resultado normal.

No llamamos al modelo al pintar cada widget ni en la presentación. El formulario controla la interacción; `predict()` controla el dominio.

## 5. Puente hacia la práctica 5.1
Ahora abre el starter oficial Churn. **No copies estos archivos sobre el starter**: allí debes completar `collect_profile`, `run_app` y `render_prediction`, conservando sus firmas y tests.

Desde la raíz del repo, en Git Bash:
```bash
cd semana5/modules/05-streamlit-basic-model-ui/exercises/01-churn-streamlit/problem/starter
uv sync
uv run python -m pytest -q
uv run streamlit run app.py
```
El README actual indica `5 failed, 8 passed` al inicio y trece tests verdes al completar la práctica. Este notebook no sustituye esa entrega. En la práctica 5.2 se transfiere el patrón a Wine y al bundle S4. En S6: `session_state`, caché, estados, latencia y telemetría.

## Fuentes
- Repo y contrato revisados el 7/10/2026: https://github.com/ssillerom/icai-muiaap-operacion-de-modelos
- Presentación S5: https://drive.google.com/file/d/1nye-s-nGzTCraNUzHR9bFK_09KO8WOIq/view?usp=drivesdk&authuser=jimenazo97%40gmail.com
- Ejecución oficial de Streamlit: https://docs.streamlit.io/develop/concepts/architecture/run-your-app

La presentación contiene referencias antiguas a un handoff Wine en 5.1. Este guion sigue la **5.1 Churn actual del repo**.